In [ ]:
# 03 Reference baseline (Experiment A)
#
# Question. What does the reference regression achieve on the frozen validation partition?
#
# Type. Formal run or inspection of Experiment A (spec). Does not train unless RUN_TRAINING = True. No official test data.
from pathlib import Path
import os

REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
os.chdir(REPO)

import dataclasses
import json
import logging

import matplotlib.pyplot as plt
import torch

from ct_iqa.config.loader import load_study_config
from ct_iqa.experiment import report
from ct_iqa.experiment.run import run_experiment
from ct_iqa.experiment.runs import read_predictions, select_run


def show(title, obj):
    """Print a section title, then the table as plain text."""
    print(f"\n=== {title} ===")
    print(obj.to_string())   # plain text table, no interactive viewer

CONFIG = "configs/experiments/a_reference.yaml"
cfg = load_study_config(CONFIG)
print(cfg.run.experiment_id, "|", cfg.model.name, "|", cfg.loss.name, "| bn_eps =", cfg.model.bn_eps,
      "| epochs =", cfg.train.epochs, "| batch =", cfg.train.batch_size, "| seed =", cfg.train.seed)


In [ ]:
# Run (explicit, expensive)
# Set RUN_TRAINING = True only with an approved compute budget; this trains for the configured number of epochs and writes a new immutable run directory.
# Training is OFF by default. Set RUN_TRAINING = True only with an approved compute budget: it trains for
# the configured number of epochs and writes a NEW immutable run directory. Progress is logged one line per epoch below the cell.
RUN_TRAINING = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RUN_ID = None      # name of a completed run to inspect; required once several completed runs exist

if RUN_TRAINING:
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", datefmt="%H:%M:%S", force=True)
    cfg = dataclasses.replace(cfg, train=dataclasses.replace(cfg.train, device=DEVICE))
    run_dir = run_experiment(cfg, config_path=CONFIG)
else:
    run_dir = select_run(cfg.run.experiment_id, RUN_ID)
print("device:", DEVICE, "| run directory:", run_dir if run_dir else "NO COMPLETED RUN FOUND (status PENDING)")


In [ ]:
# Results
if run_dir:
    history = json.loads((run_dir / "history.json").read_text())
    metrics = json.loads((run_dir / "metrics_validation.json").read_text())
    show(f"Validation metrics at selected epoch {history['best_epoch']} (policy: {history['selection_policy']}; "
         "raw 0-4 scale; selection-biased)", report.metrics_frame(metrics))
    show("Per-epoch history (selected = True marks the chosen epoch)", report.history_frame(history).round(4))
    report.plot_history(history, cfg.run.experiment_id)
    plt.show()


In [ ]:
if run_dir:
    p = read_predictions(run_dir)
    report.plot_prediction_scatter(p["label_0_4"], p["prediction_0_4"], "Validation: prediction vs label")
    plt.show()
